# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [15]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [31]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['PPCNVBBGYG', 'AAXFYXCDVI'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[16, 16,  3, 14, 22,  2,  2,  7, 25,  7],
       [ 1,  1, 24,  6, 25, 24,  3,  4, 22,  9]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  7, 25,  7,  2,  2, 22, 14,  3, 16],
       [ 0,  9, 22,  4,  3, 24, 25,  6, 24,  1]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 7, 25,  7,  2,  2, 22, 14,  3, 16, 16],
       [ 9, 22,  4,  3, 24, 25,  6, 24,  1,  1]], dtype=int32)>)


# 建立sequence to sequence 模型

In [39]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)

        self.encoder_cell = tf.keras.layers.SimpleRNNCell(128)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(128)

        self.encoder = tf.keras.layers.RNN(self.encoder_cell,
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell,
                                           return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)

    # Do not wrap call with tf.function while debugging gradient issues.
    def call(self, inputs, training=None):
        """Forward pass for Seq2Seq without attention."""
        enc_ids, dec_ids = inputs
        enc_embedded = self.embed_layer(enc_ids)
        dec_embedded = self.embed_layer(dec_ids)

        _, encoder_state = self.encoder(enc_embedded, training=training)
        decoder_outputs, _ = self.decoder(dec_embedded, initial_state=[encoder_state], training=training)
        logits = self.dense(decoder_outputs)
        return logits

    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)
        _, enc_state = self.encoder(enc_emb)
        return [enc_state]

    def get_next_token(self, x, state):
        """Single-step decoding. shape(x) = [batch]."""
        inp_emb = self.embed_layer(x)
        h, state = self.decoder_cell(inp_emb, state)
        logits = self.dense(h)
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)
        return out, state

    def initialize_model(self, input_shape):
        """Build model variables once before training."""
        batch_size, seq_len = input_shape
        encoder_input = tf.random.uniform((batch_size, seq_len), maxval=self.v_sz, dtype=tf.int32)
        decoder_input = tf.random.uniform((batch_size, seq_len), maxval=self.v_sz, dtype=tf.int32)
        _ = self([encoder_input, decoder_input], training=False)



In [40]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model([enc_x, dec_x], training=True)
        loss = compute_loss(logits, y)

    grads = tape.gradient(loss, model.trainable_variables)
    grad_var = [(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None]
    if not grad_var:
        raise ValueError('No valid gradients found. Check call() input wiring and variable creation.')

    optimizer.apply_gradients(grad_var)
    return loss

def train(model, optimizer, seqlen):
    # Build variables before training step.
    model.initialize_model((32, seqlen))

    loss = 0.0
    for step in range(3000):
        _, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss



# Loss函数以及训练逻辑

In [41]:
# 添加这行代码检查
# 创建模型
model = mySeq2SeqModel()

# 初始化模型（构建权重）
model.initialize_model((32, 20))
print("模型可训练参数数量:", len(model.trainable_variables))
print("模型参数:")
for var in model.trainable_variables:
    print(f"  {var.name}: {var.shape}")

TypeError: in user code:

    File "C:\Users\22878\AppData\Local\Temp\ipykernel_30348\537303895.py", line 74, in initialize_model  *
        _ = self((encoder_input, decoder_input))
    File "D:\Users\22878\anaconda3\envs\rl2\Lib\site-packages\keras\src\utils\traceback_utils.py", line 122, in error_handler  **
        raise e.with_traceback(filtered_tb) from None
    File "D:\Users\22878\anaconda3\envs\rl2\Lib\inspect.py", line 3212, in bind
        return self._bind(args, kwargs)
    File "D:\Users\22878\anaconda3\envs\rl2\Lib\inspect.py", line 3127, in _bind
        raise TypeError(msg) from None

    TypeError: missing a required argument: 'dec_ids'


# 训练迭代

In [29]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
model.initialize_model((32, 20))
train(model, optimizer, seqlen=20)



ValueError: in user code:

    File "C:\Users\22878\AppData\Local\Temp\ipykernel_30348\1334073275.py", line 16, in train_one_step  *
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
    File "D:\Users\22878\anaconda3\envs\rl2\Lib\site-packages\keras\src\optimizers\base_optimizer.py", line 462, in apply_gradients  **
        self.apply(grads, trainable_variables)
    File "D:\Users\22878\anaconda3\envs\rl2\Lib\site-packages\keras\src\optimizers\base_optimizer.py", line 507, in apply
        grads, trainable_variables = self._filter_empty_gradients(
    File "D:\Users\22878\anaconda3\envs\rl2\Lib\site-packages\keras\src\optimizers\base_optimizer.py", line 860, in _filter_empty_gradients
        raise ValueError("No gradients provided for any variable.")

    ValueError: No gradients provided for any variable.


# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps=10):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1]), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('OIMESIQFIQ', 'QIFQISEMIO'), ('MCFHOXNWHS', 'SHWNXOHFCM'), ('BYGBKVOAQT', 'TQAOVKBGYB'), ('RHSOVKGUWI', 'IWUGKVOSHR'), ('LKHWBGZXBO', 'OBXZGBWHKL'), ('ERYQCZPWKB', 'BKWPZCQYRE'), ('ZIQQHPBJTR', 'RTJBPHQQIZ'), ('AQFDDTPZQH', 'HQZPTDDFQA'), ('PRWJBEGJEK', 'QMJQEEJWRP'), ('MRAWVSFNZL', 'LZNFSVWARM'), ('CLFVWJRCJV', 'HQBIJWZFLC'), ('WARGSOPMKZ', 'ZKMPOSGRAW'), ('OLWBOACXQY', 'YQXCAOBWLO'), ('VBKDBZCRIP', 'PIRCZBDKBV'), ('BRWCJPWTWQ', 'QWTWPJCWRB'), ('ZARPMFTSES', 'SESTFMPRAZ'), ('VXGVVJFUBR', 'RBUFJVVGXV'), ('ETAQRUYCCQ', 'QCCYURQATE'), ('BIOVWZODCI', 'ICDOZWVOIB'), ('RLOAZAKWEP', 'PEWKAZAOLR'), ('YRMRDECTVZ', 'ZVTCEDRMRY'), ('WMPHBAHFGJ', 'JGFHABHPMW'), ('LEAHJTFHEG', 'GEHFTJHAEL'), ('WIHALQECAX', 'XACEQLAHIW'), ('XYDOZHQOTE', 'ETOQHZODYX'), ('XEVMHNNBRS', 'SRBNNHMVEX'), ('JCKCQRHZYP', 'PYZHRQCKCJ